In [ ]:
STAGE='training'
EVAL_HASH='b3b385dcdf26e99ce96a75a8c837afa87cda13169b1e17058e614607cfd0e3f7'
import csv
import gc
import hashlib
import io
import json
import os
import random
import time
from collections import defaultdict
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torchvision.ops import box_convert, box_iou

OUT = Path('/kaggle/working')
MOUNT = Path('/kaggle/input')
DIGEST = 'dfbf35a66d13e0b57447cb91b7f0e9dbc96879240ebe2a3d929f05b6bee851df'
REVISION = 'a2bb814dd30d776dcf7e30523b00659f4f141c71'
torch.set_num_threads(2)

def locate(name):
    return next(MOUNT.rglob(name))

def write_json(name, value):
    (OUT / name).write_text(json.dumps(value, indent=2))

def write_csv(name, rows):
    with (OUT / name).open('w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)

manifest_file = locate('formal_split.csv')
assert hashlib.sha256(manifest_file.read_bytes()).hexdigest() == DIGEST
with manifest_file.open() as f:
    MANIFEST = list(csv.DictReader(f))
PARTS = {s: sorted(int(r['source_id']) for r in MANIFEST if r['split'] == s) for s in ('train','validation','test')}
assert [len(PARTS[s]) for s in PARTS] == [3400,800,2000]
assert len({int(r['source_id']) for r in MANIFEST}) == 6200

def annotations():
    path = locate('instances_train2017.json')
    with path.open() as f:
        data = json.load(f)
    needed = {int(r['source_id']) for r in MANIFEST}
    sizes = {r['id']: (r['width'],r['height']) for r in data['images'] if r['id'] in needed}
    category = next(r['id'] for r in data['categories'] if r['name']=='bicycle')
    ground = defaultdict(list)
    for a in data['annotations']:
        if a['category_id'] == category and a['image_id'] in needed:
            assert not a['iscrowd']
            ground[a['image_id']].append(a['bbox'])
    result = {}
    for i in needed:
        w,h = sizes[i]
        b = torch.tensor(ground[i],dtype=torch.float32).reshape(-1,4)
        result[i] = box_convert(b / torch.tensor([w,h,w,h]),'xywh','xyxy')
    return path.parent.parent, result

def matching(overlap, scores, threshold):
    available = np.ones(overlap.shape[1],dtype=bool)
    kept = np.flatnonzero(scores >= threshold)
    order = kept[np.argsort(-scores[kept],kind='stable')]
    for qi in order:
        options = np.flatnonzero(available & (overlap[qi] >= .5))
        if len(options):
            available[options[np.argmax(overlap[qi,options])]] = False
    tp = int((~available).sum())
    return tp, len(kept)-tp, int(available.sum()), available

def checks():
    assert matching(np.zeros((2,0)),np.array([.9,.8]),.5)[:3] == (0,2,0)
    assert matching(np.array([[.8],[.9]]),np.array([.9,.8]),.5)[:3] == (1,1,0)
    assert matching(np.array([[.8,.6],[.2,.9]]),np.array([.9,.8]),.5)[:3] == (2,0,0)
    assert matching(np.array([[.8]]),np.array([.4]),.5)[:3] == (0,0,1)
    torch.manual_seed(11)
    h = head_new()
    q = torch.randn(4,256)
    assert torch.equal(h(q),torch.zeros_like(q))

def head_new():
    h = nn.Sequential(nn.Linear(256,32),nn.ReLU(),nn.Linear(32,256))
    nn.init.zeros_(h[-1].weight)
    nn.init.zeros_(h[-1].bias)
    return h

def shard_records(stage):
    paths = sorted(MOUNT.rglob(stage+'-shard-*.pt'))
    for p in paths:
        yield from torch.load(p,map_location='cpu',weights_only=True)

def prepare(rec, ground):
    i = rec['source_id']
    iou = box_iou(box_convert(rec['boxes'],'cxcywh','xyxy'),ground[i])
    labels = torch.zeros(len(iou))
    pairs = torch.nonzero(iou>=.5)
    uq, ug = set(),set()
    if len(pairs):
        labels[pairs[:,0]] = -1
        for k in torch.argsort(iou[pairs[:,0],pairs[:,1]],descending=True,stable=True).tolist():
            qi,gi = pairs[k].tolist()
            if qi not in uq and gi not in ug:
                uq.add(qi)
                ug.add(gi)
                labels[qi] = 1
    rec['iou'] = iou.numpy()
    rec['labels'] = labels
    return rec

def predict(rec, head=None):
    if head is None:
        return rec['logits'].sigmoid().numpy()
    with torch.no_grad():
        return (rec['logits'] + head(rec['queries'].float()) @ rec['text']).sigmoid().numpy()

checks()
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from threadpoolctl import threadpool_limits

started=time.perf_counter()
_,ground=annotations()
records={}
for rec in shard_records('trainval'):
    records[rec['source_id'],rec['quality']]=prepare(rec,ground)
assert set(records)=={(i,q) for s in ('train','validation') for i in PARTS[s] for q in (90,95)}

probe_report={}
for kind in ('backbone','decoder','matched'):
    datasets={}
    for split in ('train','validation'):
        x,y,groups=[],[],[]
        for i in PARTS[split]:
            pair=[records[i,q] for q in (95,90)]
            if kind=='matched':
                overlap=[r['iou'] for r in pair]
                for gi in range(len(ground[i])):
                    if all(o[:,gi].max()>=.5 for o in overlap):
                        for label,r in enumerate(pair):
                            x.append(r['queries'][r['iou'][:,gi].argmax()].float().numpy())
                            y.append(label)
                            groups.append(i)
            else:
                for label,r in enumerate(pair):
                    q=r['queries'].float()
                    value=r['backbone'] if kind=='backbone' else torch.cat([q.mean(0),q.std(0,unbiased=False)])
                    x.append(value.numpy()); y.append(label); groups.append(i)
        datasets[split]=(np.stack(x),np.array(y),np.array(groups))
    with threadpool_limits(limits=2):
        p=make_pipeline(StandardScaler(),LogisticRegression(C=1,solver='liblinear',max_iter=2000,random_state=0))
        p.fit(*datasets['train'][:2])
        x,y,groups=datasets['validation']
        correct=(p.predict(x)==y)
    ids=np.unique(groups)
    totals=np.array([correct[groups==i].sum() for i in ids])
    counts=np.array([(groups==i).sum() for i in ids])
    rng=np.random.default_rng(451)
    boot=[]
    for _ in range(20000):
        idx=rng.integers(len(ids),size=len(ids))
        boot.append(totals[idx].sum()/counts[idx].sum())
    probe_report[kind]=dict(accuracy=float(correct.mean()),ci95=np.quantile(boot,[.025,.975]).tolist(),
                            train_views=len(datasets['train'][1]),validation_views=len(y),validation_sources=len(ids))
    print('probe',kind,probe_report[kind],flush=True)
write_json('probe_report.json',probe_report)

calibration=[]
for t in np.arange(5,81,5)/100:
    totals=np.sum([matching(records[i,q]['iou'],predict(records[i,q]),t)[:3] for i in PARTS['validation'] for q in (95,90)],axis=0)
    tp,fp,fn=map(int,totals)
    calibration.append(dict(threshold=float(t),f1=2*tp/max(1,2*tp+fp+fn)))
threshold=max(calibration,key=lambda r:r['f1'])['threshold']
write_csv('formal_validation_thresholds.csv',calibration)

assignments=[]
losses=[]
hashes={}
for seed in (11,23,37):
    for name in ('A','B'):
        torch.manual_seed(seed)
        head=head_new()
        assignment={}
        rng=random.Random(seed)
        for flag in (False,True):
            ids=sorted(i for i in PARTS['train'] if bool(len(ground[i]))==flag)
            rng.shuffle(ids)
            n95=(1530 if flag else 170) if name=='A' else 850
            assignment.update({i:(95 if j<n95 else 90) for j,i in enumerate(ids)})
        assert sum(q==95 for q in assignment.values())==1700
        assignments.extend(dict(model=name,seed=seed,source_id=i,quality=q) for i,q in sorted(assignment.items()))
        optimizer=torch.optim.AdamW(head.parameters(),lr=1e-4,weight_decay=.01)
        order=random.Random(seed+771)
        for epoch in range(4):
            ids=sorted(assignment); order.shuffle(ids)
            total=0
            for begin in range(0,len(ids),6):
                batch=[records[i,assignment[i]] for i in ids[begin:begin+6]]
                queries=torch.stack([r['queries'].float() for r in batch])
                text=torch.stack([r['text'] for r in batch])
                base=torch.stack([r['logits'] for r in batch])
                labels=torch.stack([r['labels'] for r in batch])
                scores=base+torch.einsum('bqd,bd->bq',head(queries),text)
                bg=nn.functional.softplus(scores[labels==0]).mean()
                fg=scores[labels==1]
                loss=(bg+nn.functional.softplus(-fg).mean())/2 if fg.numel() else bg
                assert torch.isfinite(loss)
                optimizer.zero_grad(); loss.backward(); optimizer.step()
                total+=loss.item()
            losses.append(dict(model=name,seed=seed,epoch=epoch+1,loss=total/len(range(0,len(ids),6))))
            print(name,seed,'epoch',epoch+1,'loss',losses[-1]['loss'],flush=True)
        target=OUT/f'head-{name}-{seed}.pt'
        torch.save(head.state_dict(),target)
        hashes[target.name]=hashlib.sha256(target.read_bytes()).hexdigest()
write_csv('assignments.csv',assignments)
write_csv('training_losses.csv',losses)
write_json('frozen_training.json',dict(version=1,manifest_sha256=DIGEST,threshold=threshold,heads=hashes,
           evaluation_code_sha256=EVAL_HASH,total_seconds=time.perf_counter()-started))
